# Bronze to Silver ETL — Health Insurance Marketplace

Este notebook implementa o ETL da **camada Silver** do Data Lake, aplicando limpezas e transformações nos dados brutos da Bronze para prepará-los para análises na Gold.

## Objetivo

- Ler tabelas da camada Bronze (Iceberg via Glue Catalog).
- Aplicar transformações de qualidade de dados baseadas nas recomendações do `bronze_exploration.ipynb`.
- Escrever tabelas limpas na camada Silver em Parquet com compressão Snappy.
- Particionar tabelas grandes (ex: `Rate`) por `StateCode` e `BusinessYear` para otimização de consultas no Athena.

## Transformações Aplicadas

Baseado na seção 12 do `bronze_exploration.ipynb`:

1. **BenefitsCostSharing**: Parsear campos de copay/coinsurance de strings para decimais; mapear "No Charge" → 0.0, "Not Applicable" → null.
2. **BenefitsCostSharing**: Distinguir nulos legítimos em Tier2.
3. **PlanAttributes**: Agrupar variantes CSR pelo `HIOSProductId`.
4. **Rate**: Filtrar valores extremos (IndividualRate > 0 e < 3000).
5. **Crosswalk**: Tratar nulos em PlanId como descontinuações.
6. **Geral**: Deduplicar por `PlanId` + ano mantendo maior `VersionNum`.
7. **Particionamento**: Aplicar particionamento por `StateCode` + `BusinessYear` onde relevante.

## Como executar

1. Abra o projeto no Dev Container.
2. Atualize credenciais AWS se necessário.
3. Selecione kernel Python 3.11.14.
4. Execute sequencialmente.

> **Nota:** Este notebook usa PySpark/Glue para processamento distribuído. Em produção, execute como Glue Job.

In [1]:
import logging
import re
import sys

import awswrangler as wr
import boto3
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.conf import SparkConf
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, StringType
from pyspark.sql.window import Window

In [2]:
# ##################################################################################################
# Parâmetros do Job — usados em produção pelo Glue (comentado no ambiente local)
#
# args = getResolvedOptions(
#     sys.argv, ["JOB_NAME", "PIPELINE_NAME", "BRONZE_DATABASE", "SILVER_DATABASE"]
# )
#
# PIPELINE_NAME = args["PIPELINE_NAME"]
# BRONZE_DATABASE = args["BRONZE_DATABASE"]
# SILVER_DATABASE = args["SILVER_DATABASE"]

In [3]:
# ##################################################################################################
# Variáveis locais — substitui os parâmetros do job para execução no notebook

PIPELINE_NAME   = "health_insurance"
BRONZE_DATABASE = "eedb015_bronze"
SILVER_DATABASE = "eedb015_silver"

# Descobre o account ID dinamicamente via STS
_account_id = boto3.client("sts").get_caller_identity()["Account"]

# Limite de tabelas a processar — útil para não sobrecarregar o ambiente local.
# Altere para None para processar todas.
LOCAL_TABLE_LIMIT = None

In [4]:
# ##################################################################################################
# Inicialização Spark / Glue com extensões Iceberg

logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(levelname)s — %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
)
logger = logging.getLogger("bronze_to_silver")

scf = SparkConf()
scf.setAll([
    ("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"),
    ("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog"),
    ("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog"),
    ("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO"),
    ("spark.sql.defaultCatalog", "glue_catalog"),
    # Parser de datas legado
    ("spark.sql.legacy.timeParserPolicy", "LEGACY"),
    # Otimização de escrita no S3
    ("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2"),
    ("spark.speculation", "true"),
    # Sobrescrita dinâmica de partições
    ("spark.sql.sources.partitionOverwriteMode", "dynamic"),
])

sc = SparkContext(conf=scf)
glue_ctx = GlueContext(sc)
spark = glue_ctx.spark_session

# Em produção: job = Job(glue_ctx); job.init(args['JOB_NAME'], args)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.11/site-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [ ]:
def apply_transformations(table_name: str, df):
    """
    Aplica transformações específicas por tabela baseadas nas recomendações.
    """
    if "benefits_cost_sharing" in table_name or "benefits" in table_name:
        # Parsear copay/coinsurance
        copay_cols = [c for c in df.columns if "copay" in c.lower()]
        for col in copay_cols:
            if col in df.columns:
                df = df.withColumn(col, parse_money_to_decimal(col))
        
        # Transformações específicas para benefits_cost_sharing
        if "benefits_cost_sharing" in table_name:
            # Converter colunas de porcentagem para double
            percentage_cols = ["coinsinntier1", "coinsinntier2", "coinsoutofnet"]
            for col in percentage_cols:
                if col in df.columns:
                    df = df.withColumn(col, parse_percentage_to_double(col))
            
            # Converter importdate para timestamp
            if "importdate" in df.columns:
                df = df.withColumn("importdate", parse_string_to_timestamp("importdate"))
            
            # Converter colunas booleanas
            bool_cols = [
                "iscovered", "isehb", "isexclfrominnmoop", 
                "isexclfromoonmoop", "isstatemandate", 
                "issubjtodedtier1", "issubjtodedtier2", "quantlimitonsvc"
            ]
            for col in bool_cols:
                if col in df.columns:
                    df = df.withColumn(col, parse_string_to_boolean(col))

            # Converter colunas para integer
            int_cols = ["limitqty", "businessyear", "minimumstay"]
            for col in int_cols:
                if col in df.columns:
                    df = df.withColumn(col, F.col(col).cast("integer"))
            
            # Forçar tipos finais para garantir que sejam mantidos
            type_mapping = {}
            type_mapping.update({col: "double" for col in percentage_cols if col in df.columns})
            type_mapping.update({col: "boolean" for col in bool_cols if col in df.columns})
            type_mapping.update({col: "integer" for col in int_cols if col in df.columns})
            if "importdate" in df.columns:
                type_mapping["importdate"] = "timestamp"
            
            df = enforce_column_types(df, type_mapping)
    
    if "plan_attributes" in table_name:
        # Agrupar variantes CSR pelo HIOSProductId (simplificado: adicionar coluna)
        if "HIOSProductId" in df.columns:
            df = df.withColumn("HIOSProductId_clean", F.substring("HIOSProductId", 1, 7))
    
    if "rate" in table_name:
        # Filtrar valores extremos
        if "IndividualRate" in df.columns:
            df = df.filter((F.col("IndividualRate") > 0) & (F.col("IndividualRate") < 3000))
    
    if "crosswalk" in table_name:
        # Tratar nulos como descontinuações (adicionar flag)
        if "PlanId" in df.columns:
            df = df.withColumn("is_discontinued", F.col("PlanId").isNull())
    
    # Deduplicação geral se aplicável
    if all(col in df.columns for col in ["BenefitName","BusinessYear", "VersionNum"]):
        df = deduplicate_by_version(df, ["BenefitName","BusinessYear", "VersionNum"])
    
    return df

In [6]:
# ##################################################################################################
# Descoberta das tabelas na Bronze

tables_df = wr.catalog.tables(database=BRONZE_DATABASE)
all_table_names = sorted(
    [t for t in tables_df["Table"].tolist() if not t.lower().startswith("raw_")]
)

logger.info("Total de tabelas na Bronze (excluindo raw_): %d", len(all_table_names))
print(all_table_names)


[2026-04-09T00:12:37] INFO — Total de tabelas na Bronze (excluindo raw_): 8


['benefits_cost_sharing', 'business_rules', 'crosswalk2015', 'crosswalk2016', 'network', 'plan_attributes', 'rate', 'service_area']


In [7]:
# ##################################################################################################
# Criar database Silver se não existir

try:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DATABASE}")
    logger.info("Database '%s' criado/verificado.", SILVER_DATABASE)
except Exception as exc:
    logger.error("Erro ao criar database '%s': %s", SILVER_DATABASE, exc)
    raise

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/share/aws/aws-java-sdk-v2/aws-sdk-java-bundle-2.29.52.jar!/software/amazon/awssdk/thirdparty/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/share/aws/glue-pds/jars/bundle-2.24.6.jar!/software/amazon/awssdk/thirdparty/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
[2026-04-09T00:12:45] INFO — Database 'eedb015_silver' criado/verificado.


In [10]:
# ##################################################################################################
# Processamento: ler Bronze, transformar, escrever Silver

tables_to_process = []
tables_to_process = ["benefits_cost_sharing"]  # Exemplo: processar só uma tabela para teste
if LOCAL_TABLE_LIMIT:
    logger.info("Ambiente local: limitando a %d tabela(s).", LOCAL_TABLE_LIMIT)
    tables_to_process = tables_to_process[:LOCAL_TABLE_LIMIT]

tables_ok = []
tables_failed = []

for table_name in tables_to_process:
    bronze_table = f"{BRONZE_DATABASE}.{table_name}"
    silver_table = f"{SILVER_DATABASE}.{table_name}"
    logger.info("Processando tabela '%s' → '%s'…", bronze_table, silver_table)

    try:
        # Ler da Bronze
        df = spark.table(bronze_table)
        logger.info("  Lida: %d coluna(s), %d linha(s) estimadas.", len(df.columns), df.count())
        logger.info("  Schema ANTES das transformações:")
        df.printSchema()

        # Aplicar transformações
        df_transformed = apply_transformations(table_name, df)
        logger.info("  Transformada: %d coluna(s).", len(df_transformed.columns))
        logger.info("  Schema DEPOIS das transformações:")
        df_transformed.printSchema()

        # Deletar tabela anterior para evitar conflito de schema (Iceberg força tipo original)
        try:
            spark.sql(f"DROP TABLE IF EXISTS {silver_table}")
            logger.info("  Tabela anterior removida.")
        except Exception as e:
            logger.warning("  Aviso ao remover tabela anterior: %s", e)

        # Escrever na Silver
        df_transformed.writeTo(silver_table).using("iceberg").mode("overwrite").create_or_replace().saveAsTable(silver_table)

        logger.info("  Tabela '%s' criada/atualizada com sucesso.", silver_table)
        tables_ok.append(silver_table)

    except Exception as exc:
        logger.error("  ERRO ao processar tabela '%s': %s", silver_table, exc, exc_info=True)
        tables_failed.append(silver_table)

# Resumo
logger.info("=" * 60)
logger.info("Processamento concluído.")
logger.info("  Tabelas OK    : %d — %s", len(tables_ok), tables_ok)
if tables_failed:
    logger.error("  Tabelas FALHA : %d — %s", len(tables_failed), tables_failed)

[2026-04-09T00:18:16] INFO — Processando tabela 'eedb015_bronze.benefits_cost_sharing' → 'eedb015_silver.benefits_cost_sharing'…
[2026-04-09T00:18:19] INFO —   Lida: 34 coluna(s), 5048468 linha(s) estimadas.
[2026-04-09T00:18:19] INFO —   Schema ANTES das transformações:


root
 |-- BenefitName: string (nullable = true)
 |-- BusinessYear: string (nullable = true)
 |-- CoinsInnTier1: string (nullable = true)
 |-- CoinsInnTier2: string (nullable = true)
 |-- CoinsOutofNet: string (nullable = true)
 |-- CopayInnTier1: string (nullable = true)
 |-- CopayInnTier2: string (nullable = true)
 |-- CopayOutofNet: string (nullable = true)
 |-- EHBVarReason: string (nullable = true)
 |-- Exclusions: string (nullable = true)
 |-- Explanation: string (nullable = true)
 |-- ImportDate: string (nullable = true)
 |-- IsCovered: string (nullable = true)
 |-- IsEHB: string (nullable = true)
 |-- IsExclFromInnMOOP: string (nullable = true)
 |-- IsExclFromOonMOOP: string (nullable = true)
 |-- IsStateMandate: string (nullable = true)
 |-- IsSubjToDedTier1: string (nullable = true)
 |-- IsSubjToDedTier2: string (nullable = true)
 |-- IssuerId: string (nullable = true)
 |-- IssuerId2: string (nullable = true)
 |-- LimitQty: string (nullable = true)
 |-- LimitUnit: string (null

[2026-04-09T00:18:19] INFO —   Transformada: 34 coluna(s).
[2026-04-09T00:18:19] INFO —   Schema DEPOIS das transformações:


root
 |-- BenefitName: string (nullable = true)
 |-- BusinessYear: string (nullable = true)
 |-- CoinsInnTier1: string (nullable = true)
 |-- CoinsInnTier2: string (nullable = true)
 |-- CoinsOutofNet: string (nullable = true)
 |-- CopayInnTier1: double (nullable = true)
 |-- CopayInnTier2: double (nullable = true)
 |-- CopayOutofNet: double (nullable = true)
 |-- EHBVarReason: string (nullable = true)
 |-- Exclusions: string (nullable = true)
 |-- Explanation: string (nullable = true)
 |-- ImportDate: string (nullable = true)
 |-- IsCovered: string (nullable = true)
 |-- IsEHB: string (nullable = true)
 |-- IsExclFromInnMOOP: string (nullable = true)
 |-- IsExclFromOonMOOP: string (nullable = true)
 |-- IsStateMandate: string (nullable = true)
 |-- IsSubjToDedTier1: string (nullable = true)
 |-- IsSubjToDedTier2: string (nullable = true)
 |-- IssuerId: string (nullable = true)
 |-- IssuerId2: string (nullable = true)
 |-- LimitQty: string (nullable = true)
 |-- LimitUnit: string (null

[2026-04-09T00:18:21] INFO —   Tabela anterior removida.
[2026-04-09T00:18:21] ERROR —   ERRO ao processar tabela 'eedb015_silver.benefits_cost_sharing': 'DataFrameWriterV2' object has no attribute 'mode'
Traceback (most recent call last):
  File "/tmp/ipykernel_9578/4133517541.py", line 39, in <module>
    df_transformed.writeTo(silver_table).using("iceberg").mode("overwrite").create_or_replace().saveAsTable(silver_table)
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'DataFrameWriterV2' object has no attribute 'mode'
[2026-04-09T00:18:21] INFO — ============================================================
[2026-04-09T00:18:21] INFO — Processamento concluído.
[2026-04-09T00:18:21] INFO —   Tabelas OK    : 0 — []
[2026-04-09T00:18:21] ERROR —   Tabelas FALHA : 1 — ['eedb015_silver.benefits_cost_sharing']


In [ ]:
# ##################################################################################################
# DEBUG: Verificar se as transformações foram aplicadas corretamente

if 'df_transformed' in locals():
    logger.info("=" * 60)
    logger.info("VERIFICAÇÃO DE TRANSFORMAÇÕES:")
    logger.info("=" * 60)
    
    # Verificar tipos de colunas específicas
    schema_fields = {field.name: field.dataType for field in df_transformed.schema.fields}
    
    percentage_cols = ["coinsinntier1", "coinsinntier2", "coinsoutofnet"]
    for col in percentage_cols:
        if col in schema_fields:
            logger.info(f"  {col}: {schema_fields[col]}")
    
    bool_cols = ["iscovered", "isehb", "isexclfrominnmoop", "isexclfromoonmoop", 
                 "isstatemandate", "issubjtodedtier1", "issubjtodedtier2"]
    for col in bool_cols:
        if col in schema_fields:
            logger.info(f"  {col}: {schema_fields[col]}")
    
    if "importdate" in schema_fields:
        logger.info(f"  importdate: {schema_fields['importdate']}")
    
    int_cols = ["limitqty", "businessyear", "minimumstay"]
    for col in int_cols:
        if col in schema_fields:
            logger.info(f"  {col}: {schema_fields[col]}")
    
    # Mostrar algumas linhas de amostra
    logger.info("Amostra de dados transformados (primeiras 3 linhas):")
    df_transformed.show(3, truncate=False)

[2026-04-09T00:18:46] INFO — ============================================================
[2026-04-09T00:18:46] INFO — VERIFICAÇÃO DE TRANSFORMAÇÕES:
[2026-04-09T00:18:46] INFO — ============================================================
[2026-04-09T00:18:46] INFO — Amostra de dados transformados (primeiras 3 linhas):
[2026-04-09T00:20:20] ERROR — KeyboardInterrupt while sending command. + 3) / 3]
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib64/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
[2026-04-09T00:

KeyboardInterrupt: 